# CODEX independent model audit — v1

**Author:** CODEX — Independent Third-Party Consultant  
**Recommendation owner:** CODEX

## TL;DR
The active Engine B signal is real enough to justify continued research, but the current evidence is not promotion-comparable across all positions. Raw weights are not feature importance because active Engine B Ridge inputs were not standardized. TE's active report is in-sample, while QB/RB/WR use a fixed temporal holdout. This notebook reproduces the evidence and runs a diagnostic expanding-time technique screen; it does **not** authorize a model change.

## Context & Methods

The companion script reads the active registry, immutable model bundles, and Engine B feature table. It reproduces artifact predictions, expresses linear weights as one-standard-deviation effects, profiles feature coverage, and tests four fixed candidate techniques on 2021–2023 expanding-time outer folds. Player-cluster bootstraps quantify RMSE improvement uncertainty. Fixed hyperparameters make this a screen, not a promotion bakeoff.

In [ ]:
from pathlib import Path
import json, subprocess, sys
import pandas as pd
HERE = Path.cwd()
while not (HERE / 'app').exists():
    HERE = HERE.parent
EVIDENCE = HERE / 'docs/agent-ledger/evidence/2026-08-18'
SCRIPT = EVIDENCE / 'independent_model_audit_codex_v1.py'
RESULTS = EVIDENCE / 'independent_model_audit_codex_v1.results.json'
subprocess.run([sys.executable, str(SCRIPT), '--output', str(RESULTS)], check=True)
audit = json.loads(RESULTS.read_text())
audit['audit_status'], audit['decision_supported']

## Data

In [ ]:
pd.DataFrame([audit['data_quality']['eligible_rows_by_position']]).T.rename(columns={0: 'eligible_rows'})

In [ ]:
active_rows = []
for pos, item in audit['engine_b'].items():
    model = item['reproduced_model_metrics']
    naive = item['reproduced_naive_metrics']
    active_rows.append({
        'position': pos, 'regime': item['evaluation_regime'],
        'n': model['n'], 'model_rmse': model['rmse'], 'naive_rmse': naive['rmse'],
        'rmse_improvement_pct': 100 * (naive['rmse'] - model['rmse']) / naive['rmse']
    })
pd.DataFrame(active_rows).set_index('position')

## Results

In [ ]:
effect_rows = []
for pos, item in audit['engine_b'].items():
    for effect in item['standardized_effects'][:6]:
        effect_rows.append({'position': pos, **effect})
pd.DataFrame(effect_rows).set_index(['position', 'feature'])

In [ ]:
screen_rows = []
for pos, item in audit['technique_screen']['positions'].items():
    naive = item['pooled']['naive']['rmse']
    for technique, metric in item['pooled'].items():
        interval = metric.get('cluster_bootstrap_vs_naive', {})
        screen_rows.append({
            'position': pos, 'technique': technique, 'n': metric['n'],
            'rmse': metric['rmse'], 'spearman': metric['spearman'],
            'improvement_vs_naive_pct': 100 * (naive - metric['rmse']) / naive,
            'bootstrap_p05': interval.get('rmse_improvement_pct_p05'),
            'bootstrap_p95': interval.get('rmse_improvement_pct_p95')
        })
pd.DataFrame(screen_rows).set_index(['position', 'technique'])

## Takeaways

1. Recent PPG dominates the conditional linear effects; age and historical production provide smaller, position-dependent adjustments.
2. Scaled Ridge and Elastic Net are the most consistent first challengers. Tree methods do not establish a general advantage in this fixed diagnostic screen.
3. TE needs a comparable out-of-time validation immediately; its active in-sample report cannot establish promotion quality.
4. The formal program must replace the partial two-year-average target with explicit H1/H2/H3 horizons and model availability separately from conditional production.
5. `decision_supported` remains false until preregistered nested walk-forward gates pass and a human approves promotion.